In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# ================================================================================================
# AMAZON ML CHALLENGE 2026 — BUSINESS ENTITY RESOLUTION
# ONE-CELL FINAL PIPELINE
# AUTO-FINDS KAGGLE DATASET FILES
# ================================================================================================

import os
import re
import gc
import time
import json
import pickle
import tempfile
import unicodedata
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

T0 = time.time()

def log(msg):
    print(f"[{time.time()-T0:8.1f}s] {msg}", flush=True)

# ================================================================================================
# 0. AUTOMATICALLY FIND DATASET FILES
# ================================================================================================

SEARCH_ROOT = "/kaggle/input"

REQUIRED_FILES = [
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
    "train_ground_truth.tsv",
    "test_source1.tsv",
    "test_source2.tsv",
    "test_source3.tsv",
]

log("Searching /kaggle/input for Amazon ML dataset files...")

FOUND_FILES = {}

for base, dirs, files in os.walk(SEARCH_ROOT):
    for filename in REQUIRED_FILES:
        if filename in files:
            FOUND_FILES[filename] = os.path.join(
                base,
                filename
            )

missing_files = [
    f for f in REQUIRED_FILES
    if f not in FOUND_FILES
]

if missing_files:

    print("\n" + "=" * 90)
    print("REQUIRED DATASET FILES NOT FOUND")
    print("=" * 90)

    print("\nMissing:")
    for f in missing_files:
        print("  -", f)

    print("\nFiles currently visible under /kaggle/input:")
    for base, dirs, files in os.walk(SEARCH_ROOT):
        for f in files:
            print(os.path.join(base, f))

    raise FileNotFoundError(
        "\nAmazon ML dataset is not completely attached to this notebook."
    )

print("\n" + "=" * 90)
print("DATASET FILES FOUND")
print("=" * 90)

for f in REQUIRED_FILES:
    print(f"{f:<30} -> {FOUND_FILES[f]}")

print("=" * 90)

# ================================================================================================
# 1. CONFIG
# ================================================================================================

WORK = "/kaggle/working"

CKPT = os.path.join(
    WORK,
    "ER_FINAL_20260927"
)

BATCH_DIR = os.path.join(
    CKPT,
    "test_batches"
)

os.makedirs(CKPT, exist_ok=True)
os.makedirs(BATCH_DIR, exist_ok=True)

FINAL_OUT = os.path.join(
    WORK,
    "matching_results.tsv"
)

N_THREADS = 4
RANDOM_STATE = 42

TRAIN_S1_CHUNK = 15000
TEST_S1_BATCH = 20000

MAX_TRAIN_S1 = 400000

MAX_BLOCK_SIZE = 150
MAX_HARD_NEG_PER_S1 = 8
MAX_POS_PER_S1 = 5

TEST_SHORTLIST = 10

np.random.seed(RANDOM_STATE)

os.environ["OMP_NUM_THREADS"] = str(N_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(N_THREADS)
os.environ["MKL_NUM_THREADS"] = str(N_THREADS)

# ================================================================================================
# 2. CHECKPOINT FUNCTIONS
# ================================================================================================

def marker_path(name):
    return os.path.join(
        CKPT,
        name + ".done"
    )

def is_done(name):
    return os.path.exists(
        marker_path(name)
    )

def mark_done(name):
    with open(
        marker_path(name),
        "w"
    ) as f:
        f.write("done")

def pickle_path(name):
    return os.path.join(
        CKPT,
        name + ".pkl"
    )

def save_pickle_atomic(obj, path):

    fd, tmp = tempfile.mkstemp(
        dir=os.path.dirname(path),
        suffix=".tmp"
    )

    try:

        with os.fdopen(
            fd,
            "wb"
        ) as f:

            pickle.dump(
                obj,
                f,
                protocol=4
            )

        os.replace(
            tmp,
            path
        )

    except Exception:

        try:
            os.remove(tmp)
        except:
            pass

        raise

def save_named(obj, name):
    save_pickle_atomic(
        obj,
        pickle_path(name)
    )

def load_named(name):

    with open(
        pickle_path(name),
        "rb"
    ) as f:

        return pickle.load(f)

def named_exists(name):
    return os.path.exists(
        pickle_path(name)
    )

def force_gc():
    gc.collect()
    gc.collect()

# ================================================================================================
# 3. UNICODE-SAFE NORMALIZATION
# ================================================================================================

def normalize_text(v):

    if v is None:
        return ""

    try:

        if pd.isna(v):
            return ""

    except:
        pass

    s = unicodedata.normalize(
        "NFKC",
        str(v)
    ).casefold()

    out = []

    for ch in s:

        cat = unicodedata.category(ch)

        if (
            cat.startswith("L")
            or cat.startswith("M")
            or cat.startswith("N")
        ):
            out.append(ch)

        elif ch.isspace():
            out.append(" ")

        else:
            out.append(" ")

    s = "".join(out)

    s = re.sub(
        r"\s+",
        " ",
        s
    ).strip()

    return s

LEGAL_REPLACEMENTS = [
    (r"\bprivate\s+limited\b", "pvt ltd"),
    (r"\bprivate\s+ltd\b", "pvt ltd"),
    (r"\bincorporated\b", "inc"),
    (r"\bcorporation\b", "corp"),
    (r"\bcompany\b", "co"),
    (r"\blimited\b", "ltd"),
]

LEGAL_SUFFIXES = {
    "pvt",
    "ltd",
    "private",
    "limited",
    "llp",
    "llc",
    "inc",
    "incorporated",
    "corporation",
    "corp",
    "co",
    "company",
    "enterprises",
    "industries",
    "group",
    "gmbh",
    "sa",
    "srl",
    "plc",
}

def core_name(v):

    if not v:
        return ""

    s = v

    for pat, repl in LEGAL_REPLACEMENTS:

        s = re.sub(
            pat,
            repl,
            s
        )

    tokens = s.split()

    while (
        tokens
        and tokens[-1] in LEGAL_SUFFIXES
    ):

        tokens.pop()

    if tokens:
        return " ".join(tokens)

    return s

# ================================================================================================
# 4. ENTITY TABLE
# ================================================================================================

ENTITY_COLS = [
    "entity_id",
    "name_norm",
    "core_name",
    "addr_norm",
    "country_n",
    "name_ns",
    "name_p4",
    "core_p4",
    "addr_num",
    "addr_p4",
]

def build_entity_table(df):

    out = pd.DataFrame(
        index=df.index
    )

    out["entity_id"] = (
        df["entity_id"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    out["name_norm"] = (
        df["business_name"]
        .map(normalize_text)
    )

    out["core_name"] = (
        out["name_norm"]
        .map(core_name)
    )

    out["addr_norm"] = (
        df["business_address"]
        .map(normalize_text)
    )

    out["country_n"] = (
        df["country"]
        .map(normalize_text)
    )

    out["name_ns"] = (
        out["name_norm"]
        .str.replace(
            " ",
            "",
            regex=False
        )
    )

    out["name_p4"] = (
        out["name_ns"]
        .str.slice(0, 4)
    )

    core_ns = (
        out["core_name"]
        .str.replace(
            " ",
            "",
            regex=False
        )
    )

    out["core_p4"] = (
        core_ns
        .str.slice(0, 4)
    )

    out["addr_num"] = (
        out["addr_norm"]
        .str.replace(
            r"\D",
            "",
            regex=True
        )
        .str.slice(0, 8)
    )

    addr_ns = (
        out["addr_norm"]
        .str.replace(
            " ",
            "",
            regex=False
        )
    )

    out["addr_p4"] = (
        addr_ns
        .str.slice(0, 4)
    )

    for c in ENTITY_COLS:
        out[c] = (
            out[c]
            .fillna("")
        )

    return out.reset_index(
        drop=True
    )

# ================================================================================================
# 5. BLOCKING
# ================================================================================================

BLOCK_KEYS = [
    (
        "name_p4",
        ["name_p4", "country_n"]
    ),
    (
        "core_p4",
        ["core_p4", "country_n"]
    ),
    (
        "addr_num",
        ["addr_num", "country_n"]
    ),
    (
        "name_norm",
        ["name_norm"]
    ),
    (
        "core_name",
        ["core_name"]
    ),
]

def make_block_indexes(pool):

    indexes = {}

    for block_name, keys in BLOCK_KEYS:

        tmp = pool[
            ["entity_id"] + keys
        ].copy()

        tmp["target_idx"] = tmp.index

        for k in keys:

            tmp = tmp[
                tmp[k] != ""
            ]

        if len(tmp) == 0:

            indexes[block_name] = pd.DataFrame(
                columns=keys + ["target_idx"]
            )

            continue

        counts = (
            tmp
            .groupby(
                keys,
                sort=False
            )
            .size()
            .reset_index(
                name="_cnt"
            )
        )

        counts = counts[
            counts["_cnt"]
            <= MAX_BLOCK_SIZE
        ]

        tmp = tmp.merge(
            counts[keys],
            on=keys,
            how="inner"
        )

        indexes[block_name] = (
            tmp[
                keys + ["target_idx"]
            ]
            .reset_index(
                drop=True
            )
        )

        del tmp
        del counts

        force_gc()

        log(
            f"Block {block_name}: "
            f"{len(indexes[block_name]):,} rows"
        )

    return indexes

def candidates_for_chunk(
    left,
    block_indexes
):

    left = left.copy()

    left["s1_row"] = left.index

    frames = []

    for block_name, keys in BLOCK_KEYS:

        right = block_indexes[
            block_name
        ]

        if len(right) == 0:
            continue

        l = left[
            keys + ["s1_row"]
        ].copy()

        for k in keys:

            l = l[
                l[k] != ""
            ]

        if len(l) == 0:

            del l

            continue

        m = l.merge(
            right,
            on=keys,
            how="inner",
            sort=False
        )

        if len(m):

            frames.append(
                m[
                    [
                        "s1_row",
                        "target_idx"
                    ]
                ]
            )

        del l
        del m

        force_gc()

    if not frames:

        return pd.DataFrame(
            columns=[
                "s1_row",
                "target_idx"
            ]
        )

    cand = pd.concat(
        frames,
        ignore_index=True
    )

    cand.drop_duplicates(
        subset=[
            "s1_row",
            "target_idx"
        ],
        inplace=True
    )

    cand.reset_index(
        drop=True,
        inplace=True
    )

    del frames
    del left

    force_gc()

    return cand

# ================================================================================================
# 6. FEATURES
# ================================================================================================

FEATURE_NAMES = [
    "name_exact",
    "core_exact",
    "name_ns_exact",
    "address_exact",
    "country_same",
    "name_token_jaccard",
    "name_char_jaccard",
    "name_prefix4",
    "core_prefix4",
    "address_token_jaccard",
    "address_char_jaccard",
    "address_prefix4",
    "name_contains",
    "core_contains",
    "address_contains",
    "name_len_similarity",
    "core_len_similarity",
    "address_len_similarity",
    "number_exact",
    "block_agree",
]

def token_set(s):

    if not s:
        return set()

    return set(
        s.split()
    )

def char_ngrams(s):

    if not s:
        return set()

    if len(s) == 1:
        return {s}

    return {
        s[i:i+2]
        for i in range(
            len(s) - 1
        )
    }

def safe_jaccard_sets(a, b):

    if not a and not b:
        return 1.0

    if not a or not b:
        return 0.0

    union = len(
        a | b
    )

    if union == 0:
        return 0.0

    return len(
        a & b
    ) / union

def length_similarity(a, b):

    la = len(a)
    lb = len(b)

    if la == 0 and lb == 0:
        return 1.0

    if la == 0 or lb == 0:
        return 0.0

    return (
        1.0
        -
        abs(la - lb)
        /
        max(la, lb)
    )

def compute_features(pair_df):

    n1 = pair_df[
        "name_norm_1"
    ].tolist()

    n2 = pair_df[
        "name_norm_2"
    ].tolist()

    c1 = pair_df[
        "core_name_1"
    ].tolist()

    c2 = pair_df[
        "core_name_2"
    ].tolist()

    a1 = pair_df[
        "addr_norm_1"
    ].tolist()

    a2 = pair_df[
        "addr_norm_2"
    ].tolist()

    ns1 = pair_df[
        "name_ns_1"
    ].tolist()

    ns2 = pair_df[
        "name_ns_2"
    ].tolist()

    co1 = pair_df[
        "country_n_1"
    ].tolist()

    co2 = pair_df[
        "country_n_2"
    ].tolist()

    np1 = pair_df[
        "name_p4_1"
    ].tolist()

    np2 = pair_df[
        "name_p4_2"
    ].tolist()

    cp1 = pair_df[
        "core_p4_1"
    ].tolist()

    cp2 = pair_df[
        "core_p4_2"
    ].tolist()

    ap1 = pair_df[
        "addr_p4_1"
    ].tolist()

    ap2 = pair_df[
        "addr_p4_2"
    ].tolist()

    num1 = pair_df[
        "addr_num_1"
    ].tolist()

    num2 = pair_df[
        "addr_num_2"
    ].tolist()

    rows = []

    for i in range(
        len(pair_df)
    ):

        na = n1[i]
        nb = n2[i]

        ca = c1[i]
        cb = c2[i]

        aa = a1[i]
        ab = a2[i]

        nts1 = token_set(na)
        nts2 = token_set(nb)

        ats1 = token_set(aa)
        ats2 = token_set(ab)

        ncs1 = char_ngrams(na)
        ncs2 = char_ngrams(nb)

        acs1 = char_ngrams(aa)
        acs2 = char_ngrams(ab)

        name_exact = float(
            bool(na)
            and na == nb
        )

        core_exact = float(
            bool(ca)
            and ca == cb
        )

        ns_exact = float(
            bool(ns1[i])
            and ns1[i] == ns2[i]
        )

        addr_exact = float(
            bool(aa)
            and aa == ab
        )

        country_same = float(
            bool(co1[i])
            and co1[i] == co2[i]
        )

        name_tok = safe_jaccard_sets(
            nts1,
            nts2
        )

        name_char = safe_jaccard_sets(
            ncs1,
            ncs2
        )

        addr_tok = safe_jaccard_sets(
            ats1,
            ats2
        )

        addr_char = safe_jaccard_sets(
            acs1,
            acs2
        )

        name_contains = float(
            bool(na)
            and bool(nb)
            and (
                na in nb
                or nb in na
            )
        )

        core_contains = float(
            bool(ca)
            and bool(cb)
            and (
                ca in cb
                or cb in ca
            )
        )

        addr_contains = float(
            bool(aa)
            and bool(ab)
            and (
                aa in ab
                or ab in aa
            )
        )

        block_agree = (
            int(
                bool(np1[i])
                and np1[i] == np2[i]
            )
            +
            int(
                bool(cp1[i])
                and cp1[i] == cp2[i]
            )
            +
            int(
                bool(ap1[i])
                and ap1[i] == ap2[i]
            )
            +
            int(
                bool(num1[i])
                and num1[i] == num2[i]
            )
            +
            int(
                bool(co1[i])
                and co1[i] == co2[i]
            )
        )

        number_exact = float(
            bool(num1[i])
            and bool(num2[i])
            and num1[i] == num2[i]
        )

        rows.append([
            name_exact,
            core_exact,
            ns_exact,
            addr_exact,
            country_same,

            name_tok,
            name_char,

            float(
                bool(np1[i])
                and np1[i] == np2[i]
            ),

            float(
                bool(cp1[i])
                and cp1[i] == cp2[i]
            ),

            addr_tok,
            addr_char,

            float(
                bool(ap1[i])
                and ap1[i] == ap2[i]
            ),

            name_contains,
            core_contains,
            addr_contains,

            length_similarity(
                na,
                nb
            ),

            length_similarity(
                ca,
                cb
            ),

            length_similarity(
                aa,
                ab
            ),

            number_exact,
            float(block_agree),
        ])

    return pd.DataFrame(
        rows,
        columns=FEATURE_NAMES,
        dtype=np.float32
    )

# ================================================================================================
# 7. ATTACH PAIR DATA
# ================================================================================================

def attach_pair_data(
    cand,
    left,
    right
):

    cols = [
        "name_norm",
        "core_name",
        "addr_norm",
        "country_n",
        "name_ns",
        "name_p4",
        "core_p4",
        "addr_num",
        "addr_p4",
    ]

    l = left[
        cols
    ].copy()

    l["s1_row"] = l.index

    r = right[
        cols
    ].copy()

    r["target_idx"] = r.index

    cand = cand.merge(
        l,
        on="s1_row",
        how="left"
    )

    cand = cand.merge(
        r,
        on="target_idx",
        how="left",
        suffixes=(
            "_1",
            "_2"
        )
    )

    del l
    del r

    force_gc()

    return cand

# ================================================================================================
# 8. TRAIN CORE
# ================================================================================================

if (
    named_exists("train_core")
    and is_done("train_core")
):

    s1_train, pool_train, gt_long = (
        load_named("train_core")
    )

    log(
        "[RESUME] TRAIN core loaded."
    )

else:

    log(
        "STAGE 1 — Loading TRAIN Source 1..."
    )

    raw = pd.read_csv(
        FOUND_FILES[
            "train_source1.tsv"
        ],
        sep="\t",
        dtype=str
    )

    s1_train = build_entity_table(
        raw
    )

    del raw
    force_gc()

    log(
        f"Train Source1: "
        f"{len(s1_train):,}"
    )

    log(
        "Loading TRAIN Source 2..."
    )

    raw = pd.read_csv(
        FOUND_FILES[
            "train_source2.tsv"
        ],
        sep="\t",
        dtype=str
    )

    pool2 = build_entity_table(
        raw
    )

    del raw
    force_gc()

    log(
        f"Train Source2: "
        f"{len(pool2):,}"
    )

    log(
        "Loading TRAIN Source 3..."
    )

    raw = pd.read_csv(
        FOUND_FILES[
            "train_source3.tsv"
        ],
        sep="\t",
        dtype=str
    )

    pool3 = build_entity_table(
        raw
    )

    del raw
    force_gc()

    log(
        f"Train Source3: "
        f"{len(pool3):,}"
    )

    pool_train = pd.concat(
        [
            pool2,
            pool3
        ],
        ignore_index=True
    )

    del pool2
    del pool3

    force_gc()

    log(
        f"Train target pool: "
        f"{len(pool_train):,}"
    )

    log(
        "Loading ground truth..."
    )

    gt_raw = pd.read_csv(
        FOUND_FILES[
            "train_ground_truth.tsv"
        ],
        sep="\t",
        dtype=str
    )

    s1_lookup = pd.Series(
        s1_train.index.values,
        index=s1_train[
            "entity_id"
        ].values
    )

    target_lookup = pd.Series(
        pool_train.index.values,
        index=pool_train[
            "entity_id"
        ].values
    )

    gt = gt_raw[
        [
            "source1_entity_id",
            "matched_entity_ids"
        ]
    ].copy()

    gt[
        "matched_entity_ids"
    ] = (
        gt[
            "matched_entity_ids"
        ]
        .fillna("")
        .astype(str)
    )

    gt = (
        gt
        .assign(
            target_id=
            gt[
                "matched_entity_ids"
            ].str.split(",")
        )
        .explode(
            "target_id"
        )
    )

    gt[
        "target_id"
    ] = (
        gt[
            "target_id"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    gt = gt[
        gt["target_id"] != ""
    ]

    gt["s1_row"] = (
        gt[
            "source1_entity_id"
        ]
        .map(
            s1_lookup
        )
    )

    gt["target_idx"] = (
        gt[
            "target_id"
        ]
        .map(
            target_lookup
        )
    )

    missing = (
        gt["s1_row"].isna()
        |
        gt["target_idx"].isna()
    ).sum()

    if missing:

        log(
            f"WARNING: "
            f"{missing:,} GT mappings unresolved."
        )

    gt = gt.dropna(
        subset=[
            "s1_row",
            "target_idx"
        ]
    )

    gt["s1_row"] = (
        gt["s1_row"]
        .astype(np.int32)
    )

    gt["target_idx"] = (
        gt["target_idx"]
        .astype(np.int32)
    )

    gt_long = (
        gt[
            [
                "s1_row",
                "target_idx"
            ]
        ]
        .drop_duplicates()
        .reset_index(
            drop=True
        )
    )

    del gt_raw
    del gt
    del s1_lookup
    del target_lookup

    force_gc()

    save_named(
        (
            s1_train,
            pool_train,
            gt_long
        ),
        "train_core"
    )

    mark_done(
        "train_core"
    )

    log(
        "STAGE 1 COMPLETE."
    )

# ================================================================================================
# 9. TRAIN BLOCKS
# ================================================================================================

if (
    named_exists("train_blocks")
    and is_done("train_blocks")
):

    train_blocks = load_named(
        "train_blocks"
    )

    log(
        "[RESUME] Train blocks loaded."
    )

else:

    log(
        "STAGE 2 — Building train block indexes..."
    )

    train_blocks = make_block_indexes(
        pool_train
    )

    save_named(
        train_blocks,
        "train_blocks"
    )

    mark_done(
        "train_blocks"
    )

# ================================================================================================
# 10. SELECT TRAINING SOURCE1 ROWS
# ================================================================================================

if (
    named_exists("train_selected_rows")
    and is_done("train_selected_rows")
):

    selected_rows = load_named(
        "train_selected_rows"
    )

else:

    rng = np.random.default_rng(
        RANDOM_STATE
    )

    all_rows = np.arange(
        len(s1_train),
        dtype=np.int32
    )

    positive_rows = np.unique(
        gt_long[
            "s1_row"
        ].values
    )

    if len(all_rows) <= MAX_TRAIN_S1:

        selected_rows = all_rows

    elif len(positive_rows) >= MAX_TRAIN_S1:

        selected_rows = rng.choice(
            positive_rows,
            size=MAX_TRAIN_S1,
            replace=False
        )

    else:

        remaining = np.setdiff1d(
            all_rows,
            positive_rows
        )

        need = (
            MAX_TRAIN_S1
            - len(positive_rows)
        )

        extra = rng.choice(
            remaining,
            size=min(
                need,
                len(remaining)
            ),
            replace=False
        )

        selected_rows = np.concatenate(
            [
                positive_rows,
                extra
            ]
        )

    selected_rows = np.sort(
        selected_rows
        .astype(np.int32)
    )

    save_named(
        selected_rows,
        "train_selected_rows"
    )

    mark_done(
        "train_selected_rows"
    )

log(
    f"Training S1 rows selected: "
    f"{len(selected_rows):,}"
)

# ================================================================================================
# 11. BUILD HARD-NEGATIVE TRAINING PAIRS
# ================================================================================================

TRAIN_PAIR_FILE = os.path.join(
    CKPT,
    "training_pairs.pkl"
)

if (
    os.path.exists(TRAIN_PAIR_FILE)
    and is_done("training_pairs")
):

    with open(
        TRAIN_PAIR_FILE,
        "rb"
    ) as f:

        train_pairs = pickle.load(f)

    log(
        f"[RESUME] Training pairs: "
        f"{len(train_pairs):,}"
    )

else:

    log(
        "STAGE 3 — Building training pairs..."
    )

    gt_set = set(
        zip(
            gt_long[
                "s1_row"
            ].astype(int),

            gt_long[
                "target_idx"
            ].astype(int)
        )
    )

    gt_by_s1 = {}

    for s, t in zip(
        gt_long[
            "s1_row"
        ].values,

        gt_long[
            "target_idx"
        ].values
    ):

        s = int(s)
        t = int(t)

        gt_by_s1.setdefault(
            s,
            []
        ).append(t)

    pair_chunks = []

    total = 0

    for start in range(
        0,
        len(selected_rows),
        TRAIN_S1_CHUNK
    ):

        chunk_rows = selected_rows[
            start:
            start + TRAIN_S1_CHUNK
        ]

        left = s1_train.loc[
            chunk_rows
        ]

        cand = candidates_for_chunk(
            left,
            train_blocks
        )

        if len(cand):

            cand = cand[
                [
                    "s1_row",
                    "target_idx"
                ]
            ].drop_duplicates()

            keys = list(
                zip(
                    cand[
                        "s1_row"
                    ].astype(int),

                    cand[
                        "target_idx"
                    ].astype(int)
                )
            )

            cand["label"] = np.fromiter(
                (
                    1
                    if k in gt_set
                    else 0
                    for k in keys
                ),
                dtype=np.int8,
                count=len(keys)
            )

            forced = []

            for s in chunk_rows:

                s = int(s)

                for t in gt_by_s1.get(
                    s,
                    []
                ):

                    forced.append(
                        (
                            s,
                            int(t),
                            1
                        )
                    )

            if forced:

                forced_df = pd.DataFrame(
                    forced,
                    columns=[
                        "s1_row",
                        "target_idx",
                        "label"
                    ]
                )

                cand = pd.concat(
                    [
                        cand,
                        forced_df
                    ],
                    ignore_index=True
                )

                del forced_df

            cand.drop_duplicates(
                subset=[
                    "s1_row",
                    "target_idx"
                ],
                inplace=True
            )

            pos = cand[
                cand["label"] == 1
            ].copy()

            neg = cand[
                cand["label"] == 0
            ].copy()

            if len(pos):

                pos["_rnd"] = np.random.random(
                    len(pos)
                )

                pos = (
                    pos
                    .sort_values(
                        [
                            "s1_row",
                            "_rnd"
                        ]
                    )
                    .groupby(
                        "s1_row",
                        sort=False
                    )
                    .head(
                        MAX_POS_PER_S1
                    )
                    .drop(
                        columns="_rnd"
                    )
                )

            if len(neg):

                neg["_rnd"] = np.random.random(
                    len(neg)
                )

                neg = (
                    neg
                    .sort_values(
                        [
                            "s1_row",
                            "_rnd"
                        ]
                    )
                    .groupby(
                        "s1_row",
                        sort=False
                    )
                    .head(
                        MAX_HARD_NEG_PER_S1
                    )
                    .drop(
                        columns="_rnd"
                    )
                )

            pair_chunks.append(
                pd.concat(
                    [
                        pos,
                        neg
                    ],
                    ignore_index=True
                )
            )

            del pos
            del neg

        total += len(
            chunk_rows
        )

        del left
        del cand

        force_gc()

        if (
            total % 100000 < TRAIN_S1_CHUNK
            or total == len(selected_rows)
        ):

            log(
                f"Training progress: "
                f"{total:,}/"
                f"{len(selected_rows):,}"
            )

    train_pairs = pd.concat(
        pair_chunks,
        ignore_index=True
    )

    train_pairs.drop_duplicates(
        subset=[
            "s1_row",
            "target_idx"
        ],
        inplace=True
    )

    train_pairs["s1_row"] = (
        train_pairs[
            "s1_row"
        ].astype(np.int32)
    )

    train_pairs["target_idx"] = (
        train_pairs[
            "target_idx"
        ].astype(np.int32)
    )

    train_pairs["label"] = (
        train_pairs[
            "label"
        ].astype(np.int8)
    )

    save_pickle_atomic(
        train_pairs,
        TRAIN_PAIR_FILE
    )

    mark_done(
        "training_pairs"
    )

    del pair_chunks
    force_gc()

    log(
        f"Training pairs created: "
        f"{len(train_pairs):,}"
    )

# ================================================================================================
# 12. TRAIN FEATURES
# ================================================================================================

TRAIN_FEATURE_FILE = os.path.join(
    CKPT,
    "train_features.pkl"
)

if (
    os.path.exists(TRAIN_FEATURE_FILE)
    and is_done("train_features")
):

    with open(
        TRAIN_FEATURE_FILE,
        "rb"
    ) as f:

        train_feature_df = pickle.load(
            f
        )

    log(
        f"[RESUME] Training features: "
        f"{len(train_feature_df):,}"
    )

else:

    log(
        "STAGE 4 — Computing training features..."
    )

    feature_parts = []

    FEATURE_CHUNK = 25000

    for start in range(
        0,
        len(train_pairs),
        FEATURE_CHUNK
    ):

        p = train_pairs.iloc[
            start:
            start + FEATURE_CHUNK
        ].copy()

        p = attach_pair_data(
            p,
            s1_train,
            pool_train
        )

        Xp = compute_features(
            p
        )

        Xp["label"] = (
            p[
                "label"
            ].astype(np.int8).values
        )

        feature_parts.append(
            Xp
        )

        del p
        del Xp

        force_gc()

        if (
            start == 0
            or start + FEATURE_CHUNK
            >= len(train_pairs)
            or start % 250000 == 0
        ):

            log(
                f"Feature progress: "
                f"{min(start + FEATURE_CHUNK, len(train_pairs)):,}/"
                f"{len(train_pairs):,}"
            )

    train_feature_df = pd.concat(
        feature_parts,
        ignore_index=True
    )

    del feature_parts

    force_gc()

    save_pickle_atomic(
        train_feature_df,
        TRAIN_FEATURE_FILE
    )

    mark_done(
        "train_features"
    )

# ================================================================================================
# 13. LIGHTGBM MODEL
# ================================================================================================

MODEL_FILE = os.path.join(
    CKPT,
    "entity_resolution_final.txt"
)

META_FILE = os.path.join(
    CKPT,
    "model_meta.json"
)

import lightgbm as lgb

if (
    os.path.exists(MODEL_FILE)
    and os.path.exists(META_FILE)
    and is_done("model")
):

    model = lgb.Booster(
        model_file=MODEL_FILE
    )

    with open(
        META_FILE,
        "r"
    ) as f:

        meta = json.load(f)

    BEST_THRESHOLD = float(
        meta[
            "threshold"
        ]
    )

    log(
        f"[RESUME] Model loaded. "
        f"Threshold={BEST_THRESHOLD:.5f}"
    )

else:

    log(
        "STAGE 5 — Training LightGBM..."
    )

    from sklearn.model_selection import GroupShuffleSplit
    from sklearn.metrics import precision_recall_curve

    groups = train_pairs[
        "s1_row"
    ].values

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=RANDOM_STATE
    )

    tr_idx, va_idx = next(
        splitter.split(
            train_feature_df,
            train_feature_df["label"],
            groups=groups
        )
    )

    X_train = train_feature_df.iloc[
        tr_idx
    ][FEATURE_NAMES]

    y_train = train_feature_df.iloc[
        tr_idx
    ]["label"].values

    X_val = train_feature_df.iloc[
        va_idx
    ][FEATURE_NAMES]

    y_val = train_feature_df.iloc[
        va_idx
    ]["label"].values

    pos = int(
        (y_train == 1).sum()
    )

    neg = int(
        (y_train == 0).sum()
    )

    scale_pos = min(
        max(
            neg / max(pos, 1),
            1.0
        ),
        6.0
    )

    log(
        f"Train={len(X_train):,}, "
        f"Validation={len(X_val):,}, "
        f"Positives={pos:,}, "
        f"Negatives={neg:,}"
    )

    train_set = lgb.Dataset(
        X_train,
        label=y_train,
        free_raw_data=True
    )

    val_set = lgb.Dataset(
        X_val,
        label=y_val,
        reference=train_set,
        free_raw_data=True
    )

    params = {
        "objective": "binary",
        "metric": [
            "auc",
            "binary_logloss"
        ],

        "boosting_type": "gbdt",

        "learning_rate": 0.04,
        "num_leaves": 63,
        "max_depth": -1,

        "min_data_in_leaf": 40,

        "feature_fraction": 0.85,
        "bagging_fraction": 0.85,
        "bagging_freq": 5,

        "lambda_l1": 0.10,
        "lambda_l2": 0.50,

        "scale_pos_weight": scale_pos,

        "verbosity": -1,

        "seed": RANDOM_STATE,
        "feature_fraction_seed": RANDOM_STATE,
        "bagging_seed": RANDOM_STATE,

        "num_threads": N_THREADS,
    }

    model = lgb.train(
        params,
        train_set,
        num_boost_round=1200,
        valid_sets=[
            val_set
        ],
        valid_names=[
            "validation"
        ],
        callbacks=[
            lgb.early_stopping(80),
            lgb.log_evaluation(100)
        ]
    )

    val_pred = model.predict(
        X_val,
        num_iteration=model.best_iteration
    )

    precision, recall, thresholds = (
        precision_recall_curve(
            y_val,
            val_pred
        )
    )

    if len(thresholds):

        beta = 0.5

        f05 = (
            (1 + beta**2)
            * precision[:-1]
            * recall[:-1]
            /
            (
                beta**2
                * precision[:-1]
                + recall[:-1]
                + 1e-12
            )
        )

        best_i = int(
            np.nanargmax(
                f05
            )
        )

        BEST_THRESHOLD = float(
            thresholds[
                best_i
            ]
        )

        val_precision = float(
            precision[
                best_i
            ]
        )

        val_recall = float(
            recall[
                best_i
            ]
        )

        val_f05 = float(
            f05[
                best_i
            ]
        )

    else:

        BEST_THRESHOLD = 0.5
        val_precision = 0.0
        val_recall = 0.0
        val_f05 = 0.0

    model.save_model(
        MODEL_FILE
    )

    with open(
        META_FILE,
        "w"
    ) as f:

        json.dump(
            {
                "threshold":
                    BEST_THRESHOLD,

                "validation_precision":
                    val_precision,

                "validation_recall":
                    val_recall,

                "validation_f05":
                    val_f05,

                "best_iteration":
                    int(
                        model.best_iteration
                    ),
            },
            f,
            indent=2
        )

    mark_done(
        "model"
    )

    log(
        f"Validation F0.5: "
        f"{val_f05:.5f}"
    )

    log(
        f"Validation Precision: "
        f"{val_precision:.5f}"
    )

    log(
        f"Validation Recall: "
        f"{val_recall:.5f}"
    )

    log(
        f"Chosen threshold: "
        f"{BEST_THRESHOLD:.5f}"
    )

    del X_train
    del X_val
    del y_train
    del y_val
    del val_pred
    del train_set
    del val_set

    force_gc()

# ================================================================================================
# 14. FREE TRAINING MEMORY
# ================================================================================================

for var in [
    "s1_train",
    "pool_train",
    "gt_long",
    "train_blocks",
    "train_pairs",
    "train_feature_df",
    "selected_rows",
]:

    if var in globals():

        try:
            del globals()[var]
        except:
            pass

force_gc()

log(
    "Training memory released."
)

# ================================================================================================
# 15. LOAD TEST TABLES
# ================================================================================================

TEST_TABLE_FILE = os.path.join(
    CKPT,
    "test_tables.pkl"
)

TEST_BLOCK_FILE = os.path.join(
    CKPT,
    "test_block_indexes.pkl"
)

if (
    os.path.exists(TEST_TABLE_FILE)
    and is_done("test_tables")
):

    with open(
        TEST_TABLE_FILE,
        "rb"
    ) as f:

        test_s1, test_pool = pickle.load(
            f
        )

    log(
        "[RESUME] Test tables loaded."
    )

else:

    log(
        "STAGE 6 — Loading TEST Source 1..."
    )

    raw = pd.read_csv(
        FOUND_FILES[
            "test_source1.tsv"
        ],
        sep="\t",
        dtype=str
    )

    test_s1 = build_entity_table(
        raw
    )

    del raw
    force_gc()

    test_s1[
        "entity_id_orig"
    ] = (
        test_s1[
            "entity_id"
        ].astype(str)
    )

    log(
        f"Test Source1: "
        f"{len(test_s1):,}"
    )

    log(
        "Loading TEST Source 2..."
    )

    raw = pd.read_csv(
        FOUND_FILES[
            "test_source2.tsv"
        ],
        sep="\t",
        dtype=str
    )

    pool2 = build_entity_table(
        raw
    )

    del raw
    force_gc()

    log(
        f"Test Source2: "
        f"{len(pool2):,}"
    )

    log(
        "Loading TEST Source 3..."
    )

    raw = pd.read_csv(
        FOUND_FILES[
            "test_source3.tsv"
        ],
        sep="\t",
        dtype=str
    )

    pool3 = build_entity_table(
        raw
    )

    del raw
    force_gc()

    log(
        f"Test Source3: "
        f"{len(pool3):,}"
    )

    test_pool = pd.concat(
        [
            pool2,
            pool3
        ],
        ignore_index=True
    )

    del pool2
    del pool3

    force_gc()

    log(
        f"Test target pool: "
        f"{len(test_pool):,}"
    )

    test_blocks = make_block_indexes(
        test_pool
    )

    save_pickle_atomic(
        test_blocks,
        TEST_BLOCK_FILE
    )

    del test_blocks
    force_gc()

    save_pickle_atomic(
        (
            test_s1,
            test_pool
        ),
        TEST_TABLE_FILE
    )

    mark_done(
        "test_tables"
    )

# ================================================================================================
# 16. TEST BLOCKS
# ================================================================================================

if os.path.exists(
    TEST_BLOCK_FILE
):

    with open(
        TEST_BLOCK_FILE,
        "rb"
    ) as f:

        test_blocks = pickle.load(
            f
        )

else:

    test_blocks = make_block_indexes(
        test_pool
    )

    save_pickle_atomic(
        test_blocks,
        TEST_BLOCK_FILE
    )

    force_gc()

# ================================================================================================
# 17. EXACT UNIQUE INDEXES
# ================================================================================================

EXACT_INDEX_FILE = os.path.join(
    CKPT,
    "test_exact_indexes.pkl"
)

if os.path.exists(
    EXACT_INDEX_FILE
):

    with open(
        EXACT_INDEX_FILE,
        "rb"
    ) as f:

        exact_indexes = pickle.load(
            f
        )

else:

    log(
        "Building exact-match fallback indexes..."
    )

    exact_indexes = {}

    id_lookup = pd.Series(
        test_pool.index.values,
        index=test_pool[
            "entity_id"
        ].values
    )

    for key in [
        "name_norm",
        "core_name",
        "addr_norm"
    ]:

        tmp = test_pool[
            [
                "entity_id",
                "country_n",
                key
            ]
        ].copy()

        tmp = tmp[
            tmp[key] != ""
        ]

        counts = (
            tmp
            .groupby(
                [
                    key,
                    "country_n"
                ],
                sort=False
            )
            .size()
            .reset_index(
                name="_cnt"
            )
        )

        counts = counts[
            counts["_cnt"] == 1
        ]

        tmp = tmp.merge(
            counts[
                [
                    key,
                    "country_n"
                ]
            ],
            on=[
                key,
                "country_n"
            ],
            how="inner"
        )

        tmp["target_idx"] = (
            tmp[
                "entity_id"
            ].map(
                id_lookup
            )
        )

        exact_indexes[key] = (
            tmp[
                [
                    key,
                    "country_n",
                    "target_idx"
                ]
            ]
            .dropna()
            .copy()
        )

        del tmp
        del counts

        force_gc()

    del id_lookup

    save_pickle_atomic(
        exact_indexes,
        EXACT_INDEX_FILE
    )

# ================================================================================================
# 18. TEST INFERENCE
# ================================================================================================

target_id_array = (
    test_pool[
        "entity_id"
    ]
    .astype(str)
    .values
)

n_test = len(
    test_s1
)

n_batches = int(
    np.ceil(
        n_test /
        TEST_S1_BATCH
    )
)

log(
    f"STAGE 7 — Test inference: "
    f"{n_test:,} rows / "
    f"{n_batches:,} batches"
)

for batch_no in range(
    n_batches
):

    start = (
        batch_no
        * TEST_S1_BATCH
    )

    end = min(
        start + TEST_S1_BATCH,
        n_test
    )

    batch_file = os.path.join(
        BATCH_DIR,
        f"batch_{batch_no:05d}.pkl"
    )

    if os.path.exists(
        batch_file
    ):

        log(
            f"[RESUME] Batch "
            f"{batch_no+1}/{n_batches} already complete."
        )

        continue

    log(
        f"Processing batch "
        f"{batch_no+1}/{n_batches}: "
        f"{start:,}-{end:,}"
    )

    rows = np.arange(
        start,
        end,
        dtype=np.int32
    )

    left = test_s1.loc[
        rows
    ]

    cand = candidates_for_chunk(
        left,
        test_blocks
    )

    del left

    force_gc()

    result_pairs = []

    if len(cand):

        pair_df = attach_pair_data(
            cand,
            test_s1,
            test_pool
        )

        del cand

        force_gc()

        # --------------------------------------------------------------------
        # CHEAP PRE-SCORE
        # --------------------------------------------------------------------

        n1 = pair_df[
            "name_norm_1"
        ].tolist()

        n2 = pair_df[
            "name_norm_2"
        ].tolist()

        co1 = pair_df[
            "country_n_1"
        ].tolist()

        co2 = pair_df[
            "country_n_2"
        ].tolist()

        np1 = pair_df[
            "name_p4_1"
        ].tolist()

        np2 = pair_df[
            "name_p4_2"
        ].tolist()

        num1 = pair_df[
            "addr_num_1"
        ].tolist()

        num2 = pair_df[
            "addr_num_2"
        ].tolist()

        cheap = np.zeros(
            len(pair_df),
            dtype=np.float32
        )

        for i in range(
            len(pair_df)
        ):

            score = 0.0

            if (
                n1[i]
                and n1[i] == n2[i]
            ):

                score += 10.0

            if (
                n1[i]
                and n2[i]
                and n1[i][:4]
                == n2[i][:4]
            ):

                score += 3.0

            if (
                co1[i]
                and co1[i] == co2[i]
            ):

                score += 2.0

            if (
                num1[i]
                and num2[i]
                and num1[i] == num2[i]
            ):

                score += 3.0

            if (
                np1[i]
                and np2[i]
                and np1[i] == np2[i]
            ):

                score += 2.0

            ta = token_set(
                n1[i]
            )

            tb = token_set(
                n2[i]
            )

            if ta and tb:

                score += (
                    2.0
                    *
                    safe_jaccard_sets(
                        ta,
                        tb
                    )
                )

            cheap[i] = score

        pair_df[
            "_cheap"
        ] = cheap

        pair_df[
            "_rank"
        ] = (
            pair_df
            .groupby(
                "s1_row"
            )["_cheap"]
            .rank(
                method="first",
                ascending=False
            )
        )

        shortlist = pair_df[
            pair_df[
                "_rank"
            ] <= TEST_SHORTLIST
        ].copy()

        del pair_df

        force_gc()

        X = compute_features(
            shortlist
        )

        scores = model.predict(
            X,
            num_iteration=model.best_iteration
        )

        accepted = (
            scores
            >= BEST_THRESHOLD
        )

        if accepted.any():

            a = shortlist.loc[
                accepted,
                [
                    "s1_row",
                    "target_idx"
                ]
            ]

            result_pairs = list(
                zip(
                    a[
                        "s1_row"
                    ].astype(int),

                    a[
                        "target_idx"
                    ].astype(int)
                )
            )

            del a

        del shortlist
        del X
        del scores
        del accepted

        force_gc()

    # ============================================================================================
    # EXACT FALLBACK
    # ============================================================================================

    already = set(
        int(s)
        for s, _ in result_pairs
    )

    unmatched = [
        int(r)
        for r in rows
        if int(r) not in already
    ]

    if unmatched:

        # ------------------------------------------------------------
        # Exact name + country
        # ------------------------------------------------------------

        x = test_s1.loc[
            unmatched,
            [
                "name_norm",
                "country_n"
            ]
        ].copy()

        x["s1_row"] = x.index

        x = x[
            x[
                "name_norm"
            ] != ""
        ]

        idx = exact_indexes[
            "name_norm"
        ]

        if len(x):

            m = x.merge(
                idx,
                on=[
                    "name_norm",
                    "country_n"
                ],
                how="inner"
            )

            for s, t in zip(
                m[
                    "s1_row"
                ],
                m[
                    "target_idx"
                ]
            ):

                s = int(s)

                if s not in already:

                    result_pairs.append(
                        (
                            s,
                            int(t)
                        )
                    )

                    already.add(
                        s
                    )

            del m

        del x

        force_gc()

        # ------------------------------------------------------------
        # Exact core name + country
        # ------------------------------------------------------------

        remaining = [
            r
            for r in unmatched
            if r not in already
        ]

        if remaining:

            x = test_s1.loc[
                remaining,
                [
                    "core_name",
                    "country_n"
                ]
            ].copy()

            x["s1_row"] = x.index

            x = x[
                x[
                    "core_name"
                ] != ""
            ]

            idx = exact_indexes[
                "core_name"
            ]

            if len(x):

                m = x.merge(
                    idx,
                    on=[
                        "core_name",
                        "country_n"
                    ],
                    how="inner"
                )

                for s, t in zip(
                    m[
                        "s1_row"
                    ],
                    m[
                        "target_idx"
                    ]
                ):

                    s = int(s)

                    if s not in already:

                        result_pairs.append(
                            (
                                s,
                                int(t)
                            )
                        )

                        already.add(
                            s
                        )

                del m

            del x

            force_gc()

        # ------------------------------------------------------------
        # Exact address + country
        # ------------------------------------------------------------

        remaining = [
            r
            for r in unmatched
            if r not in already
        ]

        if remaining:

            x = test_s1.loc[
                remaining,
                [
                    "addr_norm",
                    "country_n"
                ]
            ].copy()

            x["s1_row"] = x.index

            x = x[
                x[
                    "addr_norm"
                ] != ""
            ]

            idx = exact_indexes[
                "addr_norm"
            ]

            if len(x):

                m = x.merge(
                    idx,
                    on=[
                        "addr_norm",
                        "country_n"
                    ],
                    how="inner"
                )

                for s, t in zip(
                    m[
                        "s1_row"
                    ],
                    m[
                        "target_idx"
                    ]
                ):

                    s = int(s)

                    if s not in already:

                        result_pairs.append(
                            (
                                s,
                                int(t)
                            )
                        )

                        already.add(
                            s
                        )

                del m

            del x

            force_gc()

    # ============================================================================================
    # BATCH OUTPUT
    # ============================================================================================

    if result_pairs:

        rp = pd.DataFrame(
            result_pairs,
            columns=[
                "s1_row",
                "target_idx"
            ]
        )

        rp.drop_duplicates(
            subset=[
                "s1_row",
                "target_idx"
            ],
            inplace=True
        )

        rp["target_id"] = (
            rp[
                "target_idx"
            ]
            .map(
                lambda x:
                target_id_array[
                    int(x)
                ]
            )
        )

        grouped = (
            rp
            .groupby(
                "s1_row",
                sort=False
            )[
                "target_id"
            ]
            .apply(
                lambda x:
                ",".join(
                    dict.fromkeys(
                        str(v)
                        for v in x
                    )
                )
            )
        )

    else:

        grouped = pd.Series(
            dtype=str
        )

    batch_out = pd.DataFrame({

        "source1_entity_id":
            test_s1.loc[
                rows,
                "entity_id_orig"
            ].values,

        "matched_entity_ids":
            [
                grouped.get(
                    int(r),
                    ""
                )
                for r in rows
            ]
    })

    save_pickle_atomic(
        batch_out,
        batch_file
    )

    del batch_out
    del grouped
    del result_pairs
    del rows

    if "rp" in locals():
        del rp

    force_gc()

    log(
        f"Batch "
        f"{batch_no+1}/{n_batches} COMPLETE."
    )

# ================================================================================================
# 19. FINAL OUTPUT
# ================================================================================================

log(
    "STAGE 8 — Creating final matching_results.tsv..."
)

batch_files = sorted(
    os.path.join(
        BATCH_DIR,
        f
    )
    for f in os.listdir(
        BATCH_DIR
    )
    if (
        f.startswith("batch_")
        and f.endswith(".pkl")
    )
)

if len(batch_files) != n_batches:

    raise RuntimeError(
        f"Expected {n_batches} completed batches, "
        f"but found {len(batch_files)}."
    )

parts = []

for bf in batch_files:

    with open(
        bf,
        "rb"
    ) as f:

        parts.append(
            pickle.load(f)
        )

final_df = pd.concat(
    parts,
    ignore_index=True
)

del parts

force_gc()

if len(final_df) != n_test:

    raise RuntimeError(
        f"Final row count mismatch: "
        f"{len(final_df):,} "
        f"vs "
        f"{n_test:,}"
    )

if final_df[
    "source1_entity_id"
].duplicated().any():

    raise RuntimeError(
        "Duplicate source1_entity_id detected."
    )

final_df = final_df[
    [
        "source1_entity_id",
        "matched_entity_ids"
    ]
]

final_df[
    "matched_entity_ids"
] = (
    final_df[
        "matched_entity_ids"
    ]
    .fillna("")
    .astype(str)
)

final_df.to_csv(
    FINAL_OUT,
    sep="\t",
    index=False
)

# ================================================================================================
# 20. FINAL VERIFICATION
# ================================================================================================

check = pd.read_csv(
    FINAL_OUT,
    sep="\t",
    dtype=str
)

empty = (
    check[
        "matched_entity_ids"
    ]
    .fillna("")
    .eq("")
)

match_counts = (
    check[
        "matched_entity_ids"
    ]
    .fillna("")
    .map(
        lambda x:
        0
        if not x
        else len(
            str(x).split(",")
        )
    )
)

print("\n")
print("=" * 90)
print("AMAZON ML CHALLENGE — FINAL PIPELINE COMPLETE")
print("=" * 90)

print(
    f"FINAL FILE:\n{FINAL_OUT}"
)

print(
    f"\nRows generated: "
    f"{len(check):,}"
)

print(
    f"Expected rows: "
    f"{n_test:,}"
)

print(
    f"Row count correct: "
    f"{len(check) == n_test}"
)

print(
    f"\nDuplicate Source1 IDs: "
    f"{int(check['source1_entity_id'].duplicated().sum()):,}"
)

print(
    f"No-match rows: "
    f"{int(empty.sum()):,} "
    f"({empty.mean():.2%})"
)

print(
    f"Rows with >=1 match: "
    f"{int((~empty).sum()):,}"
)

print(
    f"Rows with exactly 1 match: "
    f"{int((match_counts == 1).sum()):,}"
)

print(
    f"Rows with multiple matches: "
    f"{int((match_counts > 1).sum()):,}"
)

print("\nFirst 10 rows:")
print(
    check.head(10).to_string(
        index=False
    )
)

print("\n" + "=" * 90)
print("UPLOAD THIS FILE:")
print(FINAL_OUT)
print("=" * 90)

log("ALL DONE.")

[     0.0s] Searching /kaggle/input for Amazon ML dataset files...

DATASET FILES FOUND
train_source1.tsv              -> /kaggle/input/datasets/harshadakhajure/amazon/train_source1.tsv
train_source2.tsv              -> /kaggle/input/datasets/harshadakhajure/amazon/train_source2.tsv
train_source3.tsv              -> /kaggle/input/datasets/harshadakhajure/amazon/train_source3.tsv
train_ground_truth.tsv         -> /kaggle/input/datasets/harshadakhajure/amazon/train_ground_truth.tsv
test_source1.tsv               -> /kaggle/input/datasets/harshadakhajure/amazon/test_source1.tsv
test_source2.tsv               -> /kaggle/input/datasets/harshadakhajure/amazon/test_source2.tsv
test_source3.tsv               -> /kaggle/input/datasets/harshadakhajure/amazon/test_source3.tsv
[     0.0s] STAGE 1 — Loading TRAIN Source 1...
[   125.3s] Train Source1: 2,206,821
[   125.3s] Loading TRAIN Source 2...
[   412.3s] Train Source2: 5,034,616
[   412.3s] Loading TRAIN Source 3...
[   708.4s] Train Source3:

In [3]:
# ============================================================
# FINAL SUBMISSION VERIFICATION — DO NOT RERUN MAIN PIPELINE
# ============================================================

import os
import pandas as pd
import re

FILE = "/kaggle/working/matching_results.tsv"

print("=" * 80)
print("FINAL SUBMISSION VERIFICATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. File exists
# ------------------------------------------------------------
if not os.path.exists(FILE):
    raise FileNotFoundError(f"Submission file not found: {FILE}")

size_mb = os.path.getsize(FILE) / (1024 * 1024)

print(f"\nFile exists:              ✅")
print(f"File size:                {size_mb:.2f} MB")

# ------------------------------------------------------------
# 2. Read submission
# ------------------------------------------------------------
df = pd.read_csv(FILE, sep="\t", dtype=str, keep_default_na=False)

print(f"\nColumns:                  {list(df.columns)}")
print(f"Rows:                     {len(df):,}")

# ------------------------------------------------------------
# 3. Check required columns
# ------------------------------------------------------------
required = ["source1_entity_id", "matched_entity_ids"]

missing_cols = [c for c in required if c not in df.columns]

if missing_cols:
    print(f"\n❌ Missing columns: {missing_cols}")
else:
    print("\nRequired columns:         ✅")

# ------------------------------------------------------------
# 4. Row count
# ------------------------------------------------------------
expected_rows = 1_732_544

print(f"Expected rows:            {expected_rows:,}")
print(f"Row count correct:        {'✅ YES' if len(df) == expected_rows else '❌ NO'}")

# ------------------------------------------------------------
# 5. Duplicate Source1 IDs
# ------------------------------------------------------------
dup_count = df["source1_entity_id"].duplicated().sum()

print(f"\nDuplicate Source1 IDs:    {dup_count:,}")
print(f"Duplicate check:          {'✅ PASS' if dup_count == 0 else '❌ FAIL'}")

# ------------------------------------------------------------
# 6. Empty predictions
# ------------------------------------------------------------
matched = df["matched_entity_ids"].fillna("").astype(str)

empty_count = (matched.str.strip() == "").sum()
nonempty_count = len(df) - empty_count

print(f"\nEmpty/no-match rows:      {empty_count:,} ({empty_count/len(df)*100:.2f}%)")
print(f"Rows with matches:        {nonempty_count:,} ({nonempty_count/len(df)*100:.2f}%)")

# ------------------------------------------------------------
# 7. Check prediction ID format
# ------------------------------------------------------------
invalid_tokens = []
total_predictions = 0

for value in matched:
    value = value.strip()

    if not value:
        continue

    ids = [x.strip() for x in value.split(",") if x.strip()]
    total_predictions += len(ids)

    for entity_id in ids:
        if not re.fullmatch(r"S[23]-\d+", entity_id):
            invalid_tokens.append(entity_id)

            if len(invalid_tokens) >= 20:
                break

    if len(invalid_tokens) >= 20:
        break

print(f"\nTotal predicted target IDs: {total_predictions:,}")

if invalid_tokens:
    print("❌ Invalid prediction IDs found.")
    print("Examples:", invalid_tokens[:20])
else:
    print("Prediction ID format:     ✅ PASS")

# ------------------------------------------------------------
# 8. Check for accidental NaN / literal null strings
# ------------------------------------------------------------
bad_null_strings = matched.str.lower().isin(
    ["nan", "none", "null", "na"]
).sum()

print(f"\nLiteral bad-null values:  {bad_null_strings:,}")

if bad_null_strings == 0:
    print("Empty-value formatting:   ✅ PASS")
else:
    print("Empty-value formatting:   ❌ CHECK")

# ------------------------------------------------------------
# 9. Check duplicate target IDs within the same row
# ------------------------------------------------------------
duplicate_target_rows = 0

for value in matched:
    value = value.strip()

    if not value:
        continue

    ids = [x.strip() for x in value.split(",") if x.strip()]

    if len(ids) != len(set(ids)):
        duplicate_target_rows += 1

print(f"\nRows with duplicate IDs:  {duplicate_target_rows:,}")

if duplicate_target_rows == 0:
    print("Within-row duplicates:    ✅ PASS")
else:
    print("Within-row duplicates:    ❌ CHECK")

# ------------------------------------------------------------
# 10. Show sample exactly as stored in file
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("FIRST 15 ROWS AS ACTUALLY READ FROM TSV")
print("=" * 80)

print(df.head(15).to_string(index=False))

# ------------------------------------------------------------
# FINAL VERDICT
# ------------------------------------------------------------
all_good = (
    os.path.exists(FILE)
    and len(df) == expected_rows
    and dup_count == 0
    and not missing_cols
    and not invalid_tokens
    and bad_null_strings == 0
    and duplicate_target_rows == 0
)

print("\n" + "=" * 80)

if all_good:
    print("✅ SUBMISSION FILE PASSED BASIC STRUCTURE VERIFICATION")
    print("=" * 80)
    print("\nFile ready for submission:")
    print(FILE)
else:
    print("⚠️ VERIFICATION FOUND AN ISSUE — DO NOT UPLOAD YET")
    print("=" * 80)

FINAL SUBMISSION VERIFICATION

File exists:              ✅
File size:                60.87 MB

Columns:                  ['source1_entity_id', 'matched_entity_ids']
Rows:                     1,732,544

Required columns:         ✅
Expected rows:            1,732,544
Row count correct:        ✅ YES

Duplicate Source1 IDs:    0
Duplicate check:          ✅ PASS

Empty/no-match rows:      324,532 (18.73%)
Rows with matches:        1,408,012 (81.27%)

Total predicted target IDs: 3,194,067
Prediction ID format:     ✅ PASS

Literal bad-null values:  0
Empty-value formatting:   ✅ PASS

Rows with duplicate IDs:  0
Within-row duplicates:    ✅ PASS

FIRST 15 ROWS AS ACTUALLY READ FROM TSV
source1_entity_id                                               matched_entity_ids
     S1-714132312                                                                 
     S1-106407869                                                                 
     S1-156285671                                                

In [4]:
import os
print(os.path.exists("/kaggle/working/matching_results.tsv"))
print(os.path.getsize("/kaggle/working/matching_results.tsv") / (1024*1024), "MB")

True
60.86747074127197 MB
